In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F



In [2]:

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("mask", mask.view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)

        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.dropout(self.proj(out))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout=0.1):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size, dropout)

        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class GPT2018(nn.Module):
    def __init__(
        self,
        vocab_size,
        block_size=512,
        d_model=768,
        n_layers=12,
        n_heads=12,
        dropout=0.1,
    ):
        super().__init__()

        self.block_size = block_size

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.Sequential(
            *[
                TransformerBlock(d_model, n_heads, block_size, dropout)
                for _ in range(n_layers)
            ]
        )

        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size

        positions = torch.arange(0, T, device=idx.device)

        x = self.token_emb(idx) + self.pos_emb(positions)
        x = self.dropout(x)

        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
            )

        return logits, loss

In [ ]:
model = GPT2018(vocab_size=50000)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

model.train()

for input_ids, target_ids in dataloader:
    logits, loss = model(input_ids, target_ids)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(loss.item())

In [ ]:
class GPTClassifier(nn.Module):
    def __init__(self, gpt, d_model=768, num_classes=2):
        super().__init__()
        self.gpt = gpt
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, idx, labels=None):
        B, T = idx.shape

        positions = torch.arange(0, T, device=idx.device)
        x = self.gpt.token_emb(idx) + self.gpt.pos_emb(positions)
        x = self.gpt.dropout(x)
        x = self.gpt.blocks(x)
        x = self.gpt.ln_f(x)

        final_token = x[:, -1, :]
        logits = self.classifier(final_token)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        return logits, loss